# Build EHR Fusion Dataset — **BigQuery-native** (org/spec + labs + vitals from cloud)

Rebuilds the **EHR half** of the fusion dataset straight from **MIMIC-IV v3.1 on
BigQuery**, into the ragged data contract of `PLAN_cxr_ehr_film_fusion.md`
(§2, §5.2). No `fusion/` package; no 2.4 GB `fullmicro` CSV.

**Why cloud:** microbiology comes back *natively ragged* (`ARRAY_AGG` per
admission → all 600+ organisms as ids, no one-hot pivot), and labs/vitals are
pulled fresh. Only the things BigQuery can't give us — the frozen **CXR
embedding `img_*`**, the **CheXpert labels `y_*`**, and the **subject-level
split** — are cached once from the existing fusion CSV.

| Source | Fields |
|---|---|
| **BigQuery** `mimiciv_3_1_hosp.microbiologyevents` | `org_ids`, `spec_ids` (ragged), `n_positive_cultures` |
| **BigQuery** `mimiciv_3_1_hosp.labevents` + `d_labitems` | 10 labs, per-admission agg |
| **BigQuery** `mimiciv_3_1_icu.chartevents` + `d_items` | 6 vitals, per-ICU-stay agg |
| **CSV cache** (built once) | `img_*` [768], `y_*` [14], `split`, keys, `age/gender`, flags |

**Join key:** `(subject_id, hadm_id)` for labs/microbiology, `(subject_id,
stay_id)` for ICU vitals — matching the original SQL. Studies with no `hadm_id`
(pure ED) → org/spec = `[MISSING]`. A validation cell cross-checks the rebuilt
microbiology against the CSV's `org_*`/`spec_*` columns.


## C0 · Setup, auth, config

Colab auth for BigQuery. All feature definitions live here. Labs/vitals are
resolved by **dictionary label** (not memorized itemids) so the mapping is
auditable — the resolved itemids are printed in C3/C4.

In [ ]:
import os, pickle, numpy as np, pandas as pd

# --- BigQuery auth (Colab) ---
PROJECT     = 'mimic-ehr-v310'                     # your GCP billing project
HOSP        = 'physionet-data.mimiciv_3_1_hosp'
ICU         = 'physionet-data.mimiciv_3_1_icu'
TMP_DATASET = 'ehr_fusion_tmp'                     # scratch dataset created in PROJECT
BQ_LOCATION = 'US'                                 # physionet-data lives in US

try:
    from google.colab import auth, drive
    auth.authenticate_user()
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive'
except Exception:
    DATA_DIR = '.'
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT, location=BQ_LOCATION)

# CSV only supplies img_/y_/split/keys/age/gender/flags. The pruned ..._full.csv
# is fine here (we DROP its org_/spec_/lab_/vital_ and re-pull from BigQuery).
CSV_PATH  = os.path.join(DATA_DIR, 'fusion_dataset_hybrid_full.csv')
OUT_DIR   = os.path.join(DATA_DIR, 'ehr_fusion_bq_v1')
CACHE_DIR = os.path.join(DATA_DIR, 'cxr_cache')     # img_/y_/keys cached here once
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(CACHE_DIR, exist_ok=True)

QUICK, QUICK_N = False, 20_000
PAD, UNK, MISSING = 0, 1, 2
AGG = 'mean'                          # per-(admission|stay) aggregation of labs/vitals

# continuous EHR (re-pulled from cloud). age is from the CSV cache but joins here.
LAB_COLS  = ['lab_hemoglobin','lab_wbc','lab_rbc','lab_platelet','lab_creatinine',
             'lab_sodium','lab_potassium','lab_glucose','lab_bun','lab_chloride']
VITAL_COLS= ['vital_heart_rate','vital_spo2','vital_temperature','vital_resprate',
             'vital_sbp','vital_dbp']
DENSE_COLS= ['age','n_positive_cultures']
CONT_COLS = LAB_COLS + VITAL_COLS + DENSE_COLS
LOG_COLS  = {'lab_wbc','lab_platelet','lab_glucose','lab_creatinine','lab_bun',
             'n_positive_cultures'}          # WBC high = leukemia, not an outlier
WINSOR_PCT = (0.5, 99.5)

FLAG_SPEC = [('gender','F')]
FLAG_BIN  = ['ed_cough','smoke','cough','diabetes']

LABELS = ['Atelectasis','Cardiomegaly','Consolidation','Edema',
          'Enlarged Cardiomediastinum','Fracture','Lung Lesion','Lung Opacity',
          'No Finding','Pleural Effusion','Pleural Other','Pneumonia',
          'Pneumothorax','Support Devices']
Y_COLS = ['y_'+l for l in LABELS]
print('project', PROJECT, '| out', OUT_DIR)

## C1 · Cache the CXR side from the CSV **once**

The 768-d embedding and CheXpert labels are model/image outputs — not in MIMIC
hosp — so they come from the fusion CSV. We read it a single time, split into
`cxr_emb.npy` (streamed, flat RAM) + `keys.parquet` (keys, split, `y_*`,
`age`, `gender`, flags). On reruns this cell no-ops and loads the cache, so the
EHR rebuild afterwards is **pure BigQuery**.

In [ ]:
CXR_NPY  = os.path.join(CACHE_DIR,'cxr_emb.npy')
KEYS_PQ  = os.path.join(CACHE_DIR,'keys.parquet')
KEEP_META= ['dicom_id','subject_id','study_id','hadm_id','stay_id','study_dt',
            'split','age','gender'] + FLAG_BIN
nrows = QUICK_N if QUICK else None

if os.path.exists(CXR_NPY) and os.path.exists(KEYS_PQ) and not QUICK:
    keys = pd.read_parquet(KEYS_PQ)
    print('loaded cache:', len(keys), 'rows')
else:
    head = pd.read_csv(CSV_PATH, nrows=0); ALL = list(head.columns)
    IMG = [c for c in ALL if c.startswith('img_')]
    keys = pd.read_csv(CSV_PATH, usecols=KEEP_META+Y_COLS, nrows=nrows, low_memory=False)
    keys['split'] = keys['split'].replace({'validate':'val','valid':'val','dev':'val'})
    N = len(keys); keys.to_parquet(KEYS_PQ, index=False)
    cxr = np.lib.format.open_memmap(CXR_NPY, mode='w+', dtype=np.float32, shape=(N,768))
    done=0
    for ch in pd.read_csv(CSV_PATH, usecols=IMG, chunksize=20_000, nrows=nrows):
        ch=ch[IMG]; cxr[done:done+len(ch)]=ch.values.astype(np.float32); done+=len(ch)
        print(f'  img {done}/{N}', end='\r')
    cxr.flush(); print('\ncached', N, 'rows ->', CACHE_DIR)

# integer keys for BigQuery (nullable Int64 keeps NaN hadm_id/stay_id)
for k in ['subject_id','hadm_id','stay_id']:
    keys[k] = keys[k].astype('Int64')
keys['split'] = keys['split'].replace({'validate':'val'})
print(keys['split'].value_counts().to_dict())
print('studies w/o hadm_id (pure ED):', int(keys['hadm_id'].isna().sum()))

## C2 · Push the cohort keys to BigQuery + leakage check

Uploading `(subject_id, hadm_id, stay_id)` as a temp table lets every EHR query
**JOIN-restrict** to our studies instead of scanning all of MIMIC (cost + speed).
Also assert subject-level split disjointness (PLAN §7.2).

In [ ]:
# subject-level leakage
S = {s:set(keys.loc[keys.split==s,'subject_id'].dropna()) for s in ['train','val','test']}
for a in S:
    for b in S:
        if a<b: assert not (S[a]&S[b]), f'LEAKAGE {a}/{b}'
print('no subject-level leakage ✔')

ds = f'{PROJECT}.{TMP_DATASET}'
client.create_dataset(bigquery.Dataset(ds), exists_ok=True)
cohort = keys[['subject_id','hadm_id','stay_id']].drop_duplicates()
client.load_table_from_dataframe(
    cohort, f'{ds}.cohort',
    job_config=bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')).result()
print('cohort uploaded:', len(cohort), 'rows ->', f'{ds}.cohort')
COHORT = f'`{ds}.cohort`'

## C3 · Microbiology → ragged `org_ids` / `spec_ids` (all organisms)

`ARRAY_AGG(DISTINCT ...)` per `(subject_id, hadm_id)` returns the full multi-set
of organisms/specimens with **no one-hot pivot** — this is the native shape for
`EmbeddingBag`. Vocab is built on the **train split only**; organisms unseen in
train map to `UNK`; an admission with no positive culture → `[MISSING]`.

In [ ]:
micro = client.query(f"""
WITH pos AS (
  SELECT m.subject_id, m.hadm_id, m.org_name, m.spec_type_desc
  FROM `{HOSP}.microbiologyevents` m
  JOIN (SELECT DISTINCT subject_id, hadm_id FROM {COHORT} WHERE hadm_id IS NOT NULL) c
    USING (subject_id, hadm_id)
)
SELECT subject_id, hadm_id,
  ARRAY_AGG(DISTINCT org_name       IGNORE NULLS) AS orgs,
  ARRAY_AGG(DISTINCT spec_type_desc IGNORE NULLS) AS specs,
  COUNTIF(org_name IS NOT NULL)                   AS n_positive_cultures
FROM pos GROUP BY subject_id, hadm_id
""").to_dataframe()
micro['subject_id']=micro['subject_id'].astype('Int64'); micro['hadm_id']=micro['hadm_id'].astype('Int64')
print('micro admissions:', len(micro))
print('example orgs:', list(micro['orgs'].iloc[0])[:5] if len(micro) else None)

# attach ragged name-lists to every study by (subject_id, hadm_id)
km = keys.merge(micro, on=['subject_id','hadm_id'], how='left')
km['orgs']  = km['orgs'].apply(lambda v: list(v) if isinstance(v,(list,np.ndarray)) else [])
km['specs'] = km['specs'].apply(lambda v: list(v) if isinstance(v,(list,np.ndarray)) else [])
km['n_positive_cultures'] = km['n_positive_cultures'].fillna(0).astype(float)
train_mask = (km['split']=='train').values

def build_vocab(list_col, name):
    from collections import Counter
    cnt=Counter()
    for lst in km.loc[train_mask, list_col]: cnt.update(lst)        # train-only
    ranked=[t for t,_ in cnt.most_common()]
    name2id={t:i+3 for i,t in enumerate(ranked)}
    id2name={PAD:'<PAD>',UNK:'<UNK>',MISSING:'<MISSING>'}; id2name.update({v:k for k,v in name2id.items()})
    print(f'{name}: {len(name2id)} organisms/specimens in train (+3 reserved)')
    return {'name2id':name2id,'id2name':id2name}

org_vocab  = build_vocab('orgs','org_vocab')
spec_vocab = build_vocab('specs','spec_vocab')

def to_ids(lst, vocab):
    ids=sorted({vocab['name2id'].get(t,UNK) for t in lst})
    return ids if ids else [MISSING]
org_ids  = [to_ids(l, org_vocab)  for l in km['orgs']]
spec_ids = [to_ids(l, spec_vocab) for l in km['specs']]
print('frac studies MISSING org:', round(np.mean([o==[MISSING] for o in org_ids]),3))

## C4 · Labs → per-admission aggregate

Resolve the 10 labs by `d_labitems.label` (blood fluid), aggregate `valuenum`
per `(subject_id, hadm_id)` with `AGG`. **Note:** the original CSV's exact
lab aggregation isn't recorded, so these may differ slightly — C7 quantifies it.

In [ ]:
LAB_LABEL_SQL = """
  CASE
    WHEN LOWER(label)LIKE'%hemoglobin%' AND fluid='Blood'                 THEN 'lab_hemoglobin'
    WHEN LOWER(label)LIKE'%white blood cell%'                             THEN 'lab_wbc'
    WHEN LOWER(label)LIKE'%red blood cell%'                               THEN 'lab_rbc'
    WHEN LOWER(label)LIKE'%platelet%'                                     THEN 'lab_platelet'
    WHEN LOWER(label)='creatinine' AND fluid='Blood'                      THEN 'lab_creatinine'
    WHEN LOWER(label)='sodium' AND fluid='Blood'                          THEN 'lab_sodium'
    WHEN LOWER(label)='potassium' AND fluid='Blood'                       THEN 'lab_potassium'
    WHEN LOWER(label)='glucose' AND fluid='Blood'                         THEN 'lab_glucose'
    WHEN LOWER(label)LIKE'%urea nitrogen%' AND fluid='Blood'              THEN 'lab_bun'
    WHEN LOWER(label)='chloride' AND fluid='Blood'                        THEN 'lab_chloride'
  END"""
agg_fn = {'mean':'AVG','median':'APPROX_QUANTILES(le.valuenum,2)[OFFSET(1)]',
          'last':'ANY_VALUE(le.valuenum)'}[AGG] if AGG!='median' else 'APPROX_QUANTILES(le.valuenum,2)[OFFSET(1)]'
labs_long = client.query(f"""
WITH lab_map AS (SELECT itemid, {LAB_LABEL_SQL} AS canon FROM `{HOSP}.d_labitems`)
SELECT le.subject_id, le.hadm_id, m.canon AS lab, {agg_fn} AS val
FROM `{HOSP}.labevents` le
JOIN lab_map m USING (itemid)
JOIN (SELECT DISTINCT subject_id, hadm_id FROM {COHORT} WHERE hadm_id IS NOT NULL) c
  USING (subject_id, hadm_id)
WHERE m.canon IS NOT NULL AND le.valuenum IS NOT NULL AND le.valuenum > 0
GROUP BY le.subject_id, le.hadm_id, m.canon
""").to_dataframe()
labs_wide = labs_long.pivot_table(index=['subject_id','hadm_id'], columns='lab',
                                  values='val', aggfunc='first').reset_index()
for c in ['subject_id','hadm_id']: labs_wide[c]=labs_wide[c].astype('Int64')
print('labs resolved:', [c for c in labs_wide.columns if c.startswith('lab_')])
km = km.merge(labs_wide, on=['subject_id','hadm_id'], how='left')

## C5 · ICU vitals → per-stay aggregate

Vitals live in `mimiciv_3_1_icu.chartevents`, keyed by `stay_id`. Resolved by
`d_items.label`; temperature normalized to °C. Only studies with a `stay_id`
(ICU) get vitals — the rest stay missing (mask=0).

In [ ]:
VIT_LABEL_SQL = """
  CASE
    WHEN label='Heart Rate'                          THEN 'vital_heart_rate'
    WHEN label LIKE 'O2 saturation pulseoxymetry%'   THEN 'vital_spo2'
    WHEN label='Respiratory Rate'                    THEN 'vital_resprate'
    WHEN label IN('Non Invasive Blood Pressure systolic','Arterial Blood Pressure systolic')  THEN 'vital_sbp'
    WHEN label IN('Non Invasive Blood Pressure diastolic','Arterial Blood Pressure diastolic')THEN 'vital_dbp'
    WHEN label='Temperature Celsius'                 THEN 'vital_temperature'
    WHEN label='Temperature Fahrenheit'              THEN 'vital_temperature_f'
  END"""
vit_long = client.query(f"""
WITH vit_map AS (SELECT itemid, {VIT_LABEL_SQL} AS canon FROM `{ICU}.d_items`)
SELECT ce.subject_id, ce.stay_id, v.canon AS vital, AVG(ce.valuenum) AS val
FROM `{ICU}.chartevents` ce
JOIN vit_map v USING (itemid)
JOIN (SELECT DISTINCT subject_id, stay_id FROM {COHORT} WHERE stay_id IS NOT NULL) c
  USING (subject_id, stay_id)
WHERE v.canon IS NOT NULL AND ce.valuenum IS NOT NULL AND ce.valuenum > 0
GROUP BY ce.subject_id, ce.stay_id, v.canon
""").to_dataframe()
vw = vit_long.pivot_table(index=['subject_id','stay_id'], columns='vital',
                          values='val', aggfunc='first').reset_index()
# fold Fahrenheit into Celsius where Celsius missing
if 'vital_temperature_f' in vw:
    f=(vw['vital_temperature_f']-32)*5/9
    vw['vital_temperature']=vw.get('vital_temperature').fillna(f) if 'vital_temperature' in vw else f
    vw=vw.drop(columns=['vital_temperature_f'])
for c in ['subject_id','stay_id']: vw[c]=vw[c].astype('Int64')
print('vitals resolved:', [c for c in vw.columns if c.startswith('vital_')])
km = km.merge(vw, on=['subject_id','stay_id'], how='left')
# ensure every expected continuous column exists even if a query returned none
for c in CONT_COLS:
    if c not in km: km[c]=np.nan
print('assembled frame:', km.shape)

## C6 · Standardize continuous + flags + labels

Same contract as the CSV notebook: train-only stats, log1p on skewed labs,
winsorize, median-impute, emit `labs_mask`. Flags append with mask=1. Labels are
already 0/1.

In [ ]:
def fit_std(df, tm):
    st={}
    for c in CONT_COLS:
        raw=df[c].astype(float).values; obs=raw[tm & ~np.isnan(raw)]
        if len(obs)==0: st[c]=dict(log=False,lo=0,hi=0,median=0,mean=0,std=1); continue
        x=np.log1p(np.clip(obs,0,None)) if c in LOG_COLS else obs.copy()
        if c in LOG_COLS: lo,hi=-np.inf,np.inf
        else: lo,hi=np.percentile(x,WINSOR_PCT); x=np.clip(x,lo,hi)
        st[c]=dict(log=c in LOG_COLS,lo=float(lo),hi=float(hi),
                   median=float(np.median(x)),mean=float(x.mean()),std=float(x.std() or 1.0))
    return st
def apply_std(df, st):
    L=len(CONT_COLS); labs=np.zeros((len(df),L),np.float32); mask=np.zeros((len(df),L),np.float32)
    for j,c in enumerate(CONT_COLS):
        raw=df[c].astype(float).values; obs=~np.isnan(raw); s=st[c]
        x=np.where(obs,raw,np.nan)
        if s['log']: x=np.log1p(np.clip(x,0,None))
        x=np.clip(x,s['lo'],s['hi']); x=np.where(obs,x,s['median'])
        labs[:,j]=((x-s['mean'])/s['std']).astype(np.float32); mask[:,j]=obs.astype(np.float32)
    return labs,mask

cont_stats=fit_std(km,train_mask); cont,cont_mask=apply_std(km,cont_stats)

flag_cols_out=[f'{c}_{v}' for c,v in FLAG_SPEC]+FLAG_BIN
flags=np.zeros((len(km),len(flag_cols_out)),np.float32); k=0
for c,v in FLAG_SPEC: flags[:,k]=(km[c].astype(str).values==v).astype(np.float32); k+=1
for c in FLAG_BIN:    flags[:,k]=km[c].fillna(0).astype(float).clip(0,1).values; k+=1

labs=np.concatenate([cont,flags],1).astype(np.float32)
labs_mask=np.concatenate([cont_mask,np.ones_like(flags)],1).astype(np.float32)
LABS_COLS=CONT_COLS+flag_cols_out; L=labs.shape[1]
y=km[Y_COLS].fillna(0).astype(np.float32).values
print('labs',labs.shape,'| L',L,'| y',y.shape)
print('cont missing-rate:', dict(zip(CONT_COLS,(1-cont_mask.mean(0)).round(2))))

## C7 · Validate cloud rebuild vs the CSV (§0 sanity)

Cross-check the BigQuery microbiology against the CSV's original `org_*` columns
on the same rows: for each study, does "has ≥1 positive culture" agree? Also
compare a lab's distribution. Large disagreement ⇒ the join key or filters need
a look **before** training.

In [ ]:
try:
    chk_cols=['dicom_id']+[c for c in pd.read_csv(CSV_PATH,nrows=0).columns
                           if c.startswith('org_') or c=='lab_hemoglobin']
    csv_chk=pd.read_csv(CSV_PATH, usecols=chk_cols, nrows=(QUICK_N if QUICK else None))
    org_csv=csv_chk[[c for c in csv_chk if c.startswith('org_')]].fillna(0).astype(bool).any(axis=1).values
    org_bq =np.array([o!=[MISSING] for o in org_ids])
    agree=(org_csv==org_bq).mean()
    print(f'microbiology "has positive culture" agreement CSV vs BQ: {agree:.3f}')
    a=km['lab_hemoglobin'].astype(float); b=csv_chk['lab_hemoglobin'].astype(float)
    print(f'lab_hemoglobin  BQ mean {a.mean():.2f}  CSV mean {b.mean():.2f}  (both admission-agg)')
except Exception as e:
    print('validation skipped:', e)

## C8 · Save the new dataset

In [ ]:
np.savez_compressed(os.path.join(OUT_DIR,'ehr_arrays.npz'),labs=labs,labs_mask=labs_mask,y=y)
mo=km[['dicom_id','subject_id','study_id','hadm_id','stay_id','split']].copy()
mo['setting']=np.where(km['stay_id'].notna(),'ED_or_ICU','inpatient')
mo['org_ids']=org_ids; mo['spec_ids']=spec_ids
mo.to_parquet(os.path.join(OUT_DIR,'ehr_meta.parquet'),index=False)

import shutil
shutil.copy(os.path.join(CACHE_DIR,'cxr_emb.npy'), os.path.join(OUT_DIR,'cxr_emb.npy'))

cfg=dict(labs_cols=LABS_COLS,cont_cols=CONT_COLS,flag_cols=flag_cols_out,L=L,cxr_dim=768,
         labels=LABELS,y_cols=Y_COLS,reserved=dict(PAD=PAD,UNK=UNK,MISSING=MISSING),
         org_vocab=org_vocab,spec_vocab=spec_vocab,
         org_vocab_size=len(org_vocab['name2id'])+3,spec_vocab_size=len(spec_vocab['name2id'])+3,
         cont_stats=cont_stats,log_cols=sorted(LOG_COLS),agg=AGG,source='bigquery')
pickle.dump(cfg, open(os.path.join(OUT_DIR,'ehr_config.pkl'),'wb'))
cf=dict(class_freq=y[train_mask].sum(0).astype(int),
        neg_class_freq=(train_mask.sum()-y[train_mask].sum(0)).astype(int),
        num_train=int(train_mask.sum()))
pickle.dump(cf, open(os.path.join(OUT_DIR,'class_freq.pkl'),'wb'))
for fn in sorted(os.listdir(OUT_DIR)):
    print(' ',fn, round(os.path.getsize(os.path.join(OUT_DIR,fn))/1e6,2),'MB')

## C9 · Sanity + Torch `EmbeddingBag` consumer (PLAN §7, §5.2)

In [ ]:
cfg=pickle.load(open(os.path.join(OUT_DIR,'ehr_config.pkl'),'rb'))
mp=pd.read_parquet(os.path.join(OUT_DIR,'ehr_meta.parquet'))
ar=np.load(os.path.join(OUT_DIR,'ehr_arrays.npz')); cx=np.load(os.path.join(OUT_DIR,'cxr_emb.npy'),mmap_mode='r')
assert cx.shape==(len(mp),768) and ar['labs'].shape==(len(mp),cfg['L']) and ar['y'].shape==(len(mp),14)
assert all(len(o)>=1 for o in mp['org_ids'])
print('shapes OK · N',len(mp),'· L',cfg['L'],'· org_vocab',cfg['org_vocab_size'],'· spec_vocab',cfg['spec_vocab_size'])

import torch
from torch.utils.data import Dataset, DataLoader
class DS(Dataset):
    def __init__(s,d,split):
        s.m=pd.read_parquet(os.path.join(d,'ehr_meta.parquet')); a=np.load(os.path.join(d,'ehr_arrays.npz'))
        s.labs,s.mask,s.y=a['labs'],a['labs_mask'],a['y']; s.cx=np.load(os.path.join(d,'cxr_emb.npy'),mmap_mode='r')
        s.idx=np.where(s.m['split'].values==split)[0]
    def __len__(s): return len(s.idx)
    def __getitem__(s,k):
        i=s.idx[k]
        return dict(x_cxr=torch.from_numpy(np.asarray(s.cx[i]).copy()),
                    org=torch.tensor(s.m['org_ids'].iloc[i],dtype=torch.long),
                    spec=torch.tensor(s.m['spec_ids'].iloc[i],dtype=torch.long),
                    labs=torch.from_numpy(s.labs[i]),labs_mask=torch.from_numpy(s.mask[i]),
                    y=torch.from_numpy(s.y[i]))
def bag(seqs):
    return torch.cat(seqs), torch.tensor([0]+[len(x) for x in seqs][:-1]).cumsum(0)
def collate(b):
    return dict(x_cxr=torch.stack([x['x_cxr'] for x in b]),org=bag([x['org'] for x in b]),
                spec=bag([x['spec'] for x in b]),labs=torch.stack([x['labs'] for x in b]),
                labs_mask=torch.stack([x['labs_mask'] for x in b]),y=torch.stack([x['y'] for x in b]))
dl=DataLoader(DS(OUT_DIR,'train'),batch_size=8,shuffle=True,collate_fn=collate)
bt=next(iter(dl)); ov,oo=bt['org']
eb=torch.nn.EmbeddingBag(cfg['org_vocab_size'],16,mode='sum')
print('batch x_cxr',tuple(bt['x_cxr'].shape),'labs',tuple(bt['labs'].shape),
      'EmbeddingBag(org)->',tuple(eb(ov,oo).shape),'✔')